# 01 · API 계약과 오류 응답

FastAPI·Pydantic의 입출력 계약을 실제 요청으로 확인합니다. 권장 15분.

위에서 아래로 실행합니다. 각 실험은 새 앱을 사용하며 실제 LLM·GPU·외부 API는 기본 경로에 없습니다. 먼저 결과를 예상하고 실행한 뒤 변수 하나를 바꾸고 관찰을 기록하세요.

In [ ]:
from pathlib import Path
import sys, asyncio, json, time
from dataclasses import replace
import numpy as np
import httpx
from IPython.display import display
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from week09.lab.app import create_app
from week09.lab.service import Settings, ChatRequest, Service, TTLCache, TokenBucket, cache_key
from week09.lab.benchmark import run_load, summarize
assert sys.version_info[:2] == (3,11), 'Python 3.11 .venv 커널 선택'
OUT = ROOT/'output/validation/week09-lab'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'HTTPX',httpx.__version__)


## 입력 명세

생성 함수 호출 전에 필드·길이·타입을 검사합니다. max_tokens는 MOCK에서 토큰 수를 실제 계산하지 않으며 Ollama에만 생성 제한으로 전달합니다.

In [ ]:
display(ChatRequest.model_json_schema())
request = ChatRequest(message='반품 정책의 근거', max_tokens=64)
assert request.message == '반품 정책의 근거'

## 앱과 서버의 구분

이 커널 안에서 앱을 직접 호출합니다. 포트·네트워크·Uvicorn 성능은 측정하지 않습니다.

In [ ]:
app = create_app()
transport = httpx.ASGITransport(app=app)
print('별도 서버 없이 HTTP 계약 검사')

## 상태와 준비

alive는 프로세스, ready는 제공자 준비 확인입니다. MOCK 준비는 모델 다운로드를 뜻하지 않습니다.

In [ ]:
async with httpx.AsyncClient(transport=transport,base_url='http://lab') as client:
    health = await client.get('/health')
    ready = await client.get('/ready')
assert health.status_code == ready.status_code == 200
display(health.json()); display(ready.json())

## 정상 응답

요청 ID·버전·캐시 여부를 함께 확인합니다. 200은 의미 정확성의 증거가 아닙니다.

In [ ]:
async with httpx.AsyncClient(transport=transport,base_url='http://lab') as client:
    response = await client.post('/api/v1/chat',json=request.model_dump())
body = response.json()
assert response.status_code == 200 and body['simulated']
assert body['request_id'] == response.headers['x-request-id']
display(body)

## 빈 문자열과 공백

둘 다 의미 없는 입력이므로 거부합니다. 원문 입력을 오류 응답에 복사하지 않습니다.

In [ ]:
invalid = []
async with httpx.AsyncClient(transport=transport,base_url='http://lab') as client:
    for value in ['', '   ']:
        invalid.append(await client.post('/api/v1/chat',json={'message':value}))
assert all(r.status_code == 422 for r in invalid)
display(invalid[0].json())

## 타입과 추가 필드

숫자 문자열의 조용한 변환과 임의 필드를 허용하지 않습니다.

In [ ]:
cases = [{'message':'x','max_tokens':'64'},{'message':'x','role':'admin'},{'message':'x'*1001}]
results = []
async with httpx.AsyncClient(transport=transport,base_url='http://lab') as client:
    for case in cases:
        results.append(await client.post('/api/v1/chat',json=case))
assert all(r.status_code == 422 for r in results)
display([r.json()['error']['fields'] for r in results])

## OpenAPI 확인

브라우저 /docs와 같은 명세를 읽습니다. 프레임워크의 최신 기능 대신 고정된 0.115.14에서 지원하는 기능을 사용합니다.

In [ ]:
schema = app.openapi()
assert '/api/v1/chat' in schema['paths']
display(schema['components']['schemas']['ChatResponse']['properties'])

## 개인정보 없는 추적

메시지·답변 원문 없이 상태와 경과 시간만 기록합니다. 메모리 캐시는 답변을 보유하므로 민감 데이터에 안전한 저장소가 아닙니다.

In [ ]:
stats = list(app.state.service.traces)
assert all('message' not in r and 'output' not in r for r in stats)
display(stats)

## 상태 코드 분류

404는 경로, 422는 입력, 502는 상류 응답 문제입니다. 같은 재시도 정책을 적용하지 않습니다.

In [ ]:
async with httpx.AsyncClient(transport=transport,base_url='http://lab') as client:
    missing = await client.get('/missing')
assert missing.status_code == 404
status_plan = {'422':'입력 수정','429':'빈도 감소','503':'용량·연결 확인','504':'시간 예산·상류 확인'}
display(status_plan)

## 내 서비스 계약

자신의 기획서 요구사항을 입력·출력·실패 계약으로 옮깁니다.

In [ ]:
contract = {'input':'질문 1–1000자','output':['답변','request_id','버전'],'failure':'근거가 없으면 검토 안내','excluded':'자동 성적 결정'}
assert contract['failure'] and contract['excluded']
display(contract)

## 포트폴리오 기록

변경 변수 / 고정 조건 / 실제 관측값 / 실패 사례 / 해석의 한계 / 다음 확인을 기록하세요. 개인정보·학생 원문은 공개 저장소에 넣지 않습니다. 별도 과제 마감은 소속 공지를 따르며 이 실습에서 새 제출 기한을 만들지 않습니다.